In [1]:
import pandas as pd
import json
import os

def prepare_for_rag(df):
    """Prepares conversation data for Retrieval Augmented Generation."""

    df['Created At'] = pd.to_datetime(df['Created At'])
    df = df.sort_values(by=['Thread ID', 'Created At'])

    conversations = {}
    for _, row in df.iterrows():
        thread_id = str(row['Thread ID'])  # Ensure Thread ID is a string for dict keys
        sender_id = str(row['Sender Account ID']) if pd.notna(row['Sender Account ID']) else "Unknown Sender"
        timestamp = row['Created At'].isoformat()
        text = row['Text']

        if thread_id not in conversations:
            conversations[thread_id] = []

        conversations[thread_id].append({
            "sender": sender_id,
            "timestamp": timestamp,
            "message": text
        })
    return conversations

def save_conversations_to_json(conversations, output_file):
    """Saves the conversation data to a JSON file."""
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(conversations, f, indent=4, ensure_ascii=False)

def main():
    input_file = r"C:\Users\jatro\Dev\Dataframes\filtered_conversations.csv"  # Raw string for Windows paths
    output_file = r"C:\Users\jatro\Dev\Dataframes\RAG_conv.json"

    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found. Please check the path.")
        return

    try:
        df = pd.read_csv(input_file)
    except pd.errors.EmptyDataError:
        print(f"Error: Input file '{input_file}' is empty.")
        return
    except pd.errors.ParserError:
        print(f"Error: Could not parse '{input_file}'. Ensure it's a valid CSV.")
        return
    except UnicodeDecodeError:
        print(f"Error: UnicodeDecodeError when reading '{input_file}'. Try specifying an encoding (e.g., encoding='latin1' or encoding='utf-8-sig') in pd.read_csv().")
        return
    except Exception as e:
        print(f"An unexpected error occurred while reading the CSV: {e}")
        return

    try:
        prepared_data = prepare_for_rag(df)
        save_conversations_to_json(prepared_data, output_file)
        print(f"Conversation data prepared and saved to {output_file}")
    except KeyError as e:
        print(f"Error: Required column '{e}' not found in the CSV.")
    except Exception as e:
        print(f"An unexpected error occurred during processing: {e}")

if __name__ == "__main__":
    main()

Conversation data prepared and saved to C:\Users\jatro\Dev\Dataframes\RAG_conv.json
